# Advanced Deep Learning Framework for Multi-Class CT-Based Stroke Diagnosis
## Hemorrhagic vs Ischemic vs Normal — 5-Model Comparison + Ensemble + Grad-CAM
---
**Datasets**: Brain CT ICH + CPAISD Ischemic Stroke  
**Task**: 3-Class Classification (Normal / Hemorrhagic / Ischemic)  
**Platform**: Kaggle GPU  
**Framework**: TensorFlow 2.x / Keras


## Cell 1 — Install Dependencies

In [1]:
!pip install -q pydicom opencv-python-headless vit-keras scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.2 MB/s eta 0:00:00


## Cell 2 — Imports & Reproducibility

In [2]:
import os, random, warnings, glob, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
import pydicom
from pathlib import Path
from datetime import datetime
from scipy import stats

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    f1_score, precision_score, recall_score, accuracy_score
)

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)

print(f"TF {tf.__version__} | GPUs: {len(tf.config.list_physical_devices('GPU'))}")


2026-03-05 23:17:39.754401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772752660.001607      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772752660.071207      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772752660.660330      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772752660.660388      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772752660.660391      55 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: 2


## Cell 3 — Configuration

In [3]:
# ...existing code...
from pathlib import Path
import os

def _first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return str(Path(p).resolve())
    return None

def _find_file_any(names, roots):
    """Find first matching file from a list of names."""
    for r in roots:
        rp = Path(r)
        if not rp.exists():
            continue
        for n in names:
            f = next(rp.rglob(n), None)
            if f:
                return f
    return None

def _find_dir_any(names, roots):
    """Find first matching directory from a list of names."""
    for r in roots:
        rp = Path(r)
        if not rp.exists():
            continue
        for n in names:
            d = next((p for p in rp.rglob(n) if p.is_dir()), None)
            if d:
                return d
    return None

class Config:
    BASE_DIR = Path.cwd()

    LOCAL_CPAISD = BASE_DIR / "cpaisd"
    LOCAL_ICH    = BASE_DIR / "ncomputed"

    KAGGLE_ICH_ROOT = Path("/kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0")
    KAGGLE_CPAISD   = Path("/kaggle/input/datasets/orvile/cpaisd-acute-ischemic-stroke-dataset/dataset")

    _ich_roots = [LOCAL_ICH, KAGGLE_ICH_ROOT, BASE_DIR]

    # أكثر من احتمال لاسم CSV
    _ich_csv_file = _find_file_any(
        [
            "hemorrhage_diagnosis_raw_ct.csv",
            "hemorrhage_diagnosis.csv",
            "diagnosis.csv",
            "labels.csv",
        ],
        _ich_roots
    )
    ICH_CSV = str(_ich_csv_file) if _ich_csv_file else None

    ICH_ROOT = (
        str(_ich_csv_file.parent)
        if _ich_csv_file
        else _first_existing([LOCAL_ICH, KAGGLE_ICH_ROOT, BASE_DIR])
    )

    # أكثر من احتمال لفولدر الصور
    _ich_imgs_dir = _find_dir_any(
        ["ct_scans", "images", "image", "scans"],
        [ICH_ROOT, LOCAL_ICH, KAGGLE_ICH_ROOT, BASE_DIR]
    )
    ICH_IMGS = str(_ich_imgs_dir) if _ich_imgs_dir else str(Path(ICH_ROOT) / "ct_scans")

    _cp_candidates = [LOCAL_CPAISD / "dataset", LOCAL_CPAISD, KAGGLE_CPAISD]
    CPAISD_ROOT = _first_existing(_cp_candidates)

    WORK_DIR   = str(BASE_DIR)
    OUTPUT_DIR = str(BASE_DIR / "outputs")
    MODEL_DIR  = str(Path(OUTPUT_DIR) / "models")
    PLOT_DIR   = str(Path(OUTPUT_DIR) / "plots")
    REPORT_DIR = str(Path(OUTPUT_DIR) / "reports")
    DICOM_DIR  = str(BASE_DIR / "dicom_converted")

    IMG_SIZE   = (224, 224)
    CHANNELS   = 3

    BATCH_SIZE  = 64
    EPOCHS      = 50
    LR_STAGE1   = 3e-4
    LR_FINETUNE = 5e-6
    N_FOLDS     = 5

    CLASS_NAMES = {0: "Normal", 1: "Hemorrhagic", 2: "Ischemic"}
    N_CLASSES   = 3

    MODEL_NAMES = ["ResNet50", "DenseNet121", "EfficientNetB3", "ViT", "ConvNeXt"]

for d in [Config.OUTPUT_DIR, Config.MODEL_DIR, Config.PLOT_DIR, Config.REPORT_DIR, Config.DICOM_DIR]:
    os.makedirs(d, exist_ok=True)

print("Config ready.")
print(f"ICH root  : {Config.ICH_ROOT}")
print(f"ICH csv   : {Config.ICH_CSV}")
print(f"ICH imgs  : {Config.ICH_IMGS}")
print(f"CPAISD    : {Config.CPAISD_ROOT}")
# ...existing code...

Config ready.
ICH root  : /kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0
ICH csv   : /kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0/hemorrhage_diagnosis.csv
ICH imgs  : /kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0/ct_scans
CPAISD    : /kaggle/input/datasets/orvile/cpaisd-acute-ischemic-stroke-dataset/dataset


In [4]:
from pathlib import Path

root = Path(Config.ICH_ROOT)
for ext in ["*.png","*.jpg","*.jpeg","*.nii","*.nii.gz","*.dcm"]:
    print(ext, "->", len(list(root.rglob(ext))))

*.png -> 0
*.jpg -> 5319
*.jpeg -> 0
*.nii -> 0
*.nii.gz -> 0
*.dcm -> 0


In [5]:
# ...existing code...
def load_ich_dataset():
    """
    Load ICH dataset and assign:
      0 = Normal
      1 = Hemorrhagic
    """
    import re
    from pathlib import Path

    csv_path = Path(Config.ICH_CSV) if Config.ICH_CSV else None

    if csv_path is None or not csv_path.exists():
        csvs = list(Path(Config.ICH_ROOT).rglob("*.csv"))
        print(f"  Available CSVs: {[c.name for c in csvs]}")
        csv_path = csvs[0] if csvs else None

    label_map = {}
    if csv_path is not None and csv_path.exists():
        df_csv = pd.read_csv(csv_path)
        print(f"  ICH CSV: {csv_path}")
        print(f"  ICH CSV columns: {list(df_csv.columns)}")

        cols = {c.lower().strip(): c for c in df_csv.columns}
        pid_col = cols.get("patientnumber", cols.get("patient_number"))
        slc_col = cols.get("slicenumber", cols.get("slice_number"))
        no_hem_col = cols.get("no_hemorrhage", cols.get("nohemorrhage"))

        hem_cols = [cols[k] for k in ["intraventricular", "intraparenchymal", "subarachnoid", "epidural", "subdural"] if k in cols]

        if pid_col and slc_col and no_hem_col:
            for _, r in df_csv.iterrows():
                if pd.isna(r[pid_col]) or pd.isna(r[slc_col]):
                    continue
                p = str(int(r[pid_col]))
                s = str(int(r[slc_col]))

                if int(r[no_hem_col]) == 1:
                    lab = 0
                else:
                    lab = 1 if (sum(int(r[c]) for c in hem_cols) > 0 if hem_cols else True) else 0

                label_map[(p, s)] = int(lab)
    else:
        print("  WARNING: No ICH CSV found. Using mask fallback only.")

    img_dir = Path(Config.ICH_IMGS)
    if not img_dir.exists():
        raise RuntimeError(f"ICH_IMGS path not found: {img_dir}")

    all_images = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        all_images.extend(img_dir.rglob(ext))

    # exclude mask files from candidate images
    all_images = [p for p in all_images if "mask" not in str(p).lower()]
    print(f"  Found {len(all_images)} ICH images in: {img_dir}")

    def _safe_mask_path(p: Path):
        parts = list(p.parts)
        low = [x.lower() for x in parts]
        if "ct_scans" in low:
            i = low.index("ct_scans")
            parts[i] = "masks"
            return Path(*parts)
        return p.parent.parent / "masks" / p.name if p.parent.parent.exists() else None

    records = []
    for img_path in all_images:
        label = None

        nums = re.findall(r"\d+", f"{img_path.parent.name}_{img_path.stem}")
        if len(nums) >= 2 and label_map:
            pid, sid = str(int(nums[0])), str(int(nums[-1]))
            label = label_map.get((pid, sid))

        if label is None:
            mp = _safe_mask_path(img_path)
            label = 1 if (mp is not None and mp.exists()) else 0

        records.append({
            "filepath": str(img_path),
            "patient_id": f"ICH_{img_path.parent.name}",
            "label": int(label),
            "source": "ICH"
        })

    df = pd.DataFrame(records).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
    print(f"  ICH — Normal: {(df.label==0).sum()} | Hemorrhagic: {(df.label==1).sum()}")
    return df
# ...existing code...

In [6]:
# ...existing code...
def dicom_to_png(dcm_path, out_path, wl=40, ww=80):
    try:
        dcm   = pydicom.dcmread(str(dcm_path))
        arr   = dcm.pixel_array.astype(np.float32)
        slope = float(getattr(dcm, "RescaleSlope", 1))
        inter = float(getattr(dcm, "RescaleIntercept", 0))
        arr   = arr * slope + inter
        lo, hi = wl - ww/2, wl + ww/2
        arr   = np.clip(arr, lo, hi)
        arr   = ((arr - lo) / (hi - lo) * 255).astype(np.uint8)
        rgb   = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_path), rgb)
        return True
    except Exception:
        return False

def convert_cpaisd(cpaisd_root, out_dir):
    cpaisd_root = Path(cpaisd_root)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    dcm_files = list(cpaisd_root.rglob("*.dcm"))
    print(f"  Found {len(dcm_files)} DICOM files")
    converted = 0

    for dcm_path in dcm_files:
        if any(x in str(dcm_path).lower() for x in ["seg", "mask", "label"]):
            continue
        rel = dcm_path.relative_to(cpaisd_root)
        out_name = "__".join(rel.parts).replace(".dcm", ".png")
        out_path = out_dir / out_name
        if dicom_to_png(dcm_path, out_path):
            converted += 1

    print(f"  Converted: {converted} PNG files")
    return converted

existing = len(list(Path(Config.DICOM_DIR).glob("*.png")))
if existing < 100:
    print("Converting CPAISD DICOM → PNG ...")
    convert_cpaisd(Config.CPAISD_ROOT, Config.DICOM_DIR)
else:
    print(f"Already converted: {existing} PNGs")
# ...existing code...

Converting CPAISD DICOM → PNG ...
  Found 10165 DICOM files
  Converted: 10165 PNG files


In [7]:
# ...existing code...
from pathlib import Path

def _count_imgs(root):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff")
    n = 0
    for e in exts:
        n += len(list(Path(root).rglob(e)))
    return n

def find_best_ich_images_dir(base_root):
    base_root = Path(base_root)
    candidates = [base_root] + [p for p in base_root.rglob("*") if p.is_dir()]
    best_dir, best_n = None, 0
    for d in candidates:
        n = _count_imgs(d)
        if n > best_n:
            best_n, best_dir = n, d
    return best_dir, best_n

best_dir, best_n = find_best_ich_images_dir(Config.ICH_ROOT)
print(f"[Auto-detect] Best ICH image dir: {best_dir}")
print(f"[Auto-detect] Image count: {best_n}")

if best_dir is not None and best_n > 0:
    Config.ICH_IMGS = str(best_dir)
    print(f"[Updated] Config.ICH_IMGS = {Config.ICH_IMGS}")
else:
    print("No ICH images found under ICH_ROOT. Check dataset extraction.")
# ...existing code...

[Auto-detect] Best ICH image dir: /kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0
[Auto-detect] Image count: 5319
[Updated] Config.ICH_IMGS = /kaggle/input/datasets/vbookshelf/computed-tomography-ct-images/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.0.0


In [8]:
# ...existing code...
def load_ich_dataset():
    import re
    from pathlib import Path

    def _norm_col(c):
        return re.sub(r"[^a-z0-9]", "", c.lower())

    csv_path = Path(Config.ICH_CSV) if Config.ICH_CSV else None
    if csv_path is None or not csv_path.exists():
        csvs = list(Path(Config.ICH_ROOT).rglob("*.csv"))
        csv_path = csvs[0] if csvs else None

    has_csv = csv_path is not None and csv_path.exists()
    label_map = {}

    if has_csv:
        df_csv = pd.read_csv(csv_path)
        col_by_norm = {_norm_col(c): c for c in df_csv.columns}
        pid_col = col_by_norm.get("patientnumber")
        sid_col = col_by_norm.get("slicenumber")
        noh_col = col_by_norm.get("nohemorrhage")
        if not (pid_col and sid_col and noh_col):
            raise RuntimeError("ICH CSV missing required columns: PatientNumber, SliceNumber, No_Hemorrhage")

        for _, r in df_csv.iterrows():
            if pd.isna(r[pid_col]) or pd.isna(r[sid_col]) or pd.isna(r[noh_col]):
                continue
            pid, sid = int(r[pid_col]), int(r[sid_col])
            label_map[(pid, sid)] = 0 if int(r[noh_col]) == 1 else 1

    img_dir = Path(Config.ICH_IMGS)
    if not img_dir.exists():
        raise RuntimeError(f"ICH_IMGS path not found: {img_dir}")

    imgs = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        imgs.extend(img_dir.rglob(ext))
    imgs = [p for p in imgs if "mask" not in str(p).lower()]

    def candidates(p: Path):
        nums = [int(x) for x in re.findall(r"\d+", str(p))]
        out = []
        try:
            out.append((int(p.parent.name), int(p.stem)))
        except Exception:
            pass
        for i in range(len(nums)-1):
            out.append((nums[i], nums[i+1]))
        if len(nums) >= 2:
            out.append((nums[-2], nums[-1]))
        # unique keep-order
        seen, uniq = set(), []
        for t in out:
            if t not in seen:
                seen.add(t)
                uniq.append(t)
        return uniq

    records, matched = [], 0
    for p in imgs:
        label, pid_used = None, None

        if has_csv:
            for pid, sid in candidates(p):
                v = label_map.get((pid, sid))
                if v is not None:
                    label, pid_used = int(v), pid
                    matched += 1
                    break
        else:
            mp = p.parent.parent / "masks" / p.name if p.parent.parent.exists() else None
            label = 1 if (mp is not None and mp.exists()) else 0

        if label is None:
            continue

        records.append({
            "filepath": str(p),
            "patient_id": f"ICH_{pid_used if pid_used is not None else p.parent.name}",
            "label": int(label),
            "source": "ICH"
        })

    df = pd.DataFrame(records).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
    print(f"ICH total={len(df)} | Normal={(df.label==0).sum()} | Hemorrhagic={(df.label==1).sum()} | matched={matched}")

    if (df.label == 1).sum() == 0:
        raise RuntimeError("Hemorrhagic class is zero. Check ICH path/CSV mapping.")
    return df


def load_ischemic_dataset():
    pngs = sorted(Path(Config.DICOM_DIR).rglob("*.png"))
    if not pngs:
        raise RuntimeError("No PNGs in DICOM_DIR — re-run DICOM conversion cell.")

    records = []
    for i, p in enumerate(pngs):
        parts = p.stem.split("__")
        pid = parts[1] if len(parts) > 1 else f"P{i//30:03d}"
        records.append({
            "filepath": str(p),
            "patient_id": f"ISC_{pid}",
            "label": 2,
            "source": "CPAISD"
        })
    df = pd.DataFrame(records).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
    print(f"CPAISD total={len(df)} | Ischemic={(df.label==2).sum()} | patients={df.patient_id.nunique()}")
    return df


# build df_all مرة واحدة فقط
df_ich = load_ich_dataset()
df_ischemic = load_ischemic_dataset()
df_all = pd.concat([df_ich, df_ischemic], ignore_index=True).drop_duplicates(subset=["filepath"]).reset_index(drop=True)

print("df_all shape:", df_all.shape)
print(df_all["label"].value_counts().sort_index())

required = {0, 1, 2}
present = set(df_all["label"].unique().tolist())
missing = required - present
if missing:
    raise RuntimeError(f"Missing classes before split: {missing}")
# ...existing code...

ICH total=5319 | Normal=4365 | Hemorrhagic=954 | matched=5319
CPAISD total=10165 | Ischemic=10165 | patients=112
df_all shape: (15484, 4)
label
0     4365
1      954
2    10165
Name: count, dtype: int64


In [10]:
# ...existing code...
# Backward-compatible aliases for old cells
ICH_ROOT     = Config.ICH_ROOT
ICH_CSV      = Config.ICH_CSV
ICH_IMGS     = Config.ICH_IMGS
CPAISD_ROOT  = Config.CPAISD_ROOT
DICOM_DIR    = Config.DICOM_DIR
OUTPUT_DIR   = Config.OUTPUT_DIR
MODEL_DIR    = Config.MODEL_DIR
PLOT_DIR     = Config.PLOT_DIR
REPORT_DIR   = Config.REPORT_DIR

IMG_SIZE     = Config.IMG_SIZE
BATCH_SIZE   = Config.BATCH_SIZE
EPOCHS       = Config.EPOCHS
LR_STAGE1    = Config.LR_STAGE1
LR_FINETUNE  = Config.LR_FINETUNE
N_FOLDS      = Config.N_FOLDS
N_CLASSES    = Config.N_CLASSES
CLASS_NAMES  = Config.CLASS_NAMES
MODEL_NAMES  = Config.MODEL_NAMES
# ...existing code...

In [11]:
def patient_wise_split(df, test_frac=0.20, val_frac=0.10):
    pl    = df.groupby("patient_id")["label"].agg(
                lambda x: x.mode()[0]).reset_index()
    pids, plabs = pl.patient_id.values, pl.label.values

    def can_stratify(y):
        vc = pd.Series(y).value_counts()
        return len(vc) > 1 and vc.min() >= 2

    p_train, p_temp, _, l_temp = train_test_split(
        pids, plabs, test_size=test_frac+val_frac,
        stratify=plabs if can_stratify(plabs) else None,
        random_state=SEED)

    vf = val_frac / (val_frac + test_frac)
    p_val, p_test = train_test_split(
        p_temp, test_size=1-vf,
        stratify=l_temp if can_stratify(l_temp) else None,
        random_state=SEED)

    tr = df[df.patient_id.isin(p_train)].reset_index(drop=True)
    vl = df[df.patient_id.isin(p_val)].reset_index(drop=True)
    te = df[df.patient_id.isin(p_test)].reset_index(drop=True)

    print(f"  Train:{len(tr)} | Val:{len(vl)} | Test:{len(te)}")
    for cls, name in CLASS_NAMES.items():
        print(f"    {name}: Train={( tr.label==cls).sum()} "
              f"Val={(vl.label==cls).sum()} Test={(te.label==cls).sum()}")
    return tr, vl, te


def oversample(df):
    """Oversample minority classes to match majority. Training only."""
    n_max    = df.label.value_counts().max()
    balanced = [df]
    for cls in df.label.unique():
        n = (df.label == cls).sum()
        if n < n_max:
            extra = df[df.label==cls].sample(
                n=n_max-n, replace=True, random_state=SEED)
            balanced.append(extra)
    df_bal = pd.concat(balanced, ignore_index=True).sample(
        frac=1, random_state=SEED).reset_index(drop=True)
    print(f"  After oversample: {len(df_bal)} "
          f"({' | '.join(f'{CLASS_NAMES[c]}={( df_bal.label==c).sum()}' for c in CLASS_NAMES)})")
    return df_bal


print("[Splitting]")
train_df, val_df, test_df = patient_wise_split(df_all)

print("\n[Balancing train set]")
train_df_bal = oversample(train_df)

labs = train_df.label.values
cw   = compute_class_weight("balanced", classes=np.unique(labs), y=labs)
class_weights = dict(zip(np.unique(labs).astype(int), cw))
print(f"\nClass weights: {class_weights}")


[Splitting]
  Train:10939 | Val:1366 | Test:3179
    Normal: Train=3037 Val=430 Test=898
    Hemorrhagic: Train=597 Val=129 Test=228
    Ischemic: Train=7305 Val=807 Test=2053

[Balancing train set]
  After oversample: 21915 (Normal=7305 | Hemorrhagic=7305 | Ischemic=7305)

Class weights: {np.int64(0): np.float64(1.2006365931291845), np.int64(1): np.float64(6.107761027359017), np.int64(2): np.float64(0.4991558293406343)}


In [12]:
# ...existing code...
def load_ischemic_dataset():
    pngs = sorted(Path(Config.DICOM_DIR).rglob("*.png"))
    if not pngs:
        raise RuntimeError("No PNGs in DICOM_DIR — re-run DICOM conversion cell.")

    records = []
    for i, p in enumerate(pngs):
        parts = p.stem.split("__")
        pid = parts[1] if len(parts) > 1 else f"P{i//30:03d}"
        records.append({
            "filepath": str(p),
            "patient_id": f"ISC_{pid}",
            "label": 2,
            "source": "CPAISD"
        })

    df = pd.DataFrame(records).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
    print(f"  CPAISD — Ischemic: {len(df)} | Patients: {df.patient_id.nunique()}")
    return df

def oversample_to_balance(df):
    return oversample(df)
# ...existing code...

In [13]:
# ...existing code...
if "df_all" in globals():
    del df_all

df_ich = load_ich_dataset()
df_ischemic = load_ischemic_dataset()

df_all = pd.concat([df_ich, df_ischemic], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["filepath"]).reset_index(drop=True)

print(f"df_all shape: {df_all.shape}")
print(df_all["label"].value_counts(dropna=False).sort_index())

required = {0, 1, 2}
present = set(df_all["label"].unique().tolist())
missing = required - present
if missing:
    raise RuntimeError(f"Missing classes before split: {missing}")
# ...existing code...

ICH total=5319 | Normal=4365 | Hemorrhagic=954 | matched=5319
  CPAISD — Ischemic: 10165 | Patients: 112
df_all shape: (15484, 4)
label
0     4365
1      954
2    10165
Name: count, dtype: int64


In [14]:
# ...existing code...
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# =========================
# 1) Split + class weights
# =========================
def patient_wise_split(df, test_frac=0.20, val_frac=0.10, seed=42):
    pl = df.groupby("patient_id")["label"].agg(lambda x: x.mode()[0]).reset_index()
    pids, plabs = pl.patient_id.values, pl.label.values

    p_train, p_temp, _, l_temp = train_test_split(
        pids, plabs, test_size=test_frac + val_frac, stratify=plabs, random_state=seed
    )
    vf = val_frac / (val_frac + test_frac)
    p_val, p_test = train_test_split(p_temp, test_size=1 - vf, stratify=l_temp, random_state=seed)

    tr = df[df.patient_id.isin(p_train)].reset_index(drop=True)
    vl = df[df.patient_id.isin(p_val)].reset_index(drop=True)
    te = df[df.patient_id.isin(p_test)].reset_index(drop=True)

    print(f"Train={len(tr)} | Val={len(vl)} | Test={len(te)}")
    print("Train class counts:\n", tr.label.value_counts().sort_index())
    return tr, vl, te

train_df, val_df, test_df = patient_wise_split(df_all, test_frac=0.20, val_frac=0.10, seed=SEED)

cw = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(train_df.label.unique())),
    y=train_df.label.values
)
class_weights = dict(zip(sorted(train_df.label.unique()), cw))
print("class_weights:", class_weights)

# =========================
# 2) tf.data (خفيف للـ CT)
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomContrast(0.08),
], name="ct_aug")

def parse_fn(fp, label, training=False):
    img = tf.io.read_file(fp)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if training:
        img = aug(img, training=True)
    return img, tf.cast(label, tf.int32)

def make_ds(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df.filepath.values, df.label.values))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: parse_fn(x, y, training), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# =========================
# 3) Architectures (model-by-model)
# =========================
def cls_head(x, n_classes=3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    return layers.Dense(n_classes, activation="softmax")(x)

def build_densenet121(n_classes=3, lr=3e-4):
    base = tf.keras.applications.DenseNet121(
        include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False
    inp = Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.densenet.preprocess_input(inp * 255.0)
    x = base(x, training=False)
    out = cls_head(x, n_classes)
    model = Model(inp, out, name="DenseNet121_v2")
    model.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model, base

def build_efficientnetb0(n_classes=3, lr=3e-4):
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False
    inp = Input(shape=IMG_SIZE + (3,))
    x = base(inp * 255.0, training=False)  # EfficientNet internal rescaling
    out = cls_head(x, n_classes)
    model = Model(inp, out, name="EfficientNetB0_v2")
    model.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model, base

def build_resnet50v2(n_classes=3, lr=3e-4):
    base = tf.keras.applications.ResNet50V2(
        include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False
    inp = Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.resnet_v2.preprocess_input(inp * 255.0)
    x = base(x, training=False)
    out = cls_head(x, n_classes)
    model = Model(inp, out, name="ResNet50V2_v2")
    model.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model, base

BUILDERS = {
    "DenseNet121": build_densenet121,
    "EfficientNetB0": build_efficientnetb0,
    "ResNet50V2": build_resnet50v2
}

# =========================
# 4) Train engine (stage1 + fine-tune)
# =========================
def train_model_v2(model_name, e1=12, e2=6, unfreeze_last=30, lr1=3e-4, lr2=1e-5):
    train_ds = make_ds(train_df, training=True)
    val_ds   = make_ds(val_df,   training=False)
    test_ds  = make_ds(test_df,  training=False)

    model, base = BUILDERS[model_name](n_classes=3, lr=lr1)

    cbs = [
        EarlyStopping(monitor="val_accuracy", mode="max", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7)
    ]

    print(f"\n[{model_name}] Stage-1")
    model.fit(train_ds, validation_data=val_ds, epochs=e1, class_weight=class_weights, callbacks=cbs, verbose=1)

    # fine-tune
    base.trainable = True
    for l in base.layers[:-unfreeze_last]:
        l.trainable = False

    model.compile(optimizer=Adam(lr2), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print(f"[{model_name}] Stage-2 Fine-tune")
    model.fit(train_ds, validation_data=val_ds, epochs=e2, class_weight=class_weights, callbacks=cbs, verbose=1)

    # evaluate
    y_true, y_pred = [], []
    for x, y in test_ds:
        p = model.predict(x, verbose=0).argmax(axis=1)
        y_true.extend(y.numpy())
        y_pred.extend(p)

    print(f"\n[{model_name}] Test ACC={accuracy_score(y_true, y_pred):.4f} | F1(macro)={f1_score(y_true, y_pred, average='macro'):.4f}")
    print(classification_report(y_true, y_pred, target_names=["Normal","Hemorrhagic","Ischemic"]))
    return model

# ...existing code...

Train=10939 | Val=1366 | Test=3179
Train class counts:
 label
0    3037
1     597
2    7305
Name: count, dtype: int64
class_weights: {np.int64(0): np.float64(1.2006365931291845), np.int64(1): np.float64(6.107761027359017), np.int64(2): np.float64(0.4991558293406343)}


I0000 00:00:1772753091.166424      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772753091.172457      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [15]:
m1 = train_model_v2("DenseNet121",    e1=14, e2=8, unfreeze_last=40)
m2 = train_model_v2("EfficientNetB0", e1=12, e2=6, unfreeze_last=30)
m3 = train_model_v2("ResNet50V2",     e1=12, e2=6, unfreeze_last=30)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

[DenseNet121] Stage-1
Epoch 1/14


I0000 00:00:1772753114.083237     140 service.cc:152] XLA service 0x7cf218006230 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772753114.083282     140 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1772753114.083287     140 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1772753118.018544     140 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/342 ━━━━━━━━━━━━━━━━━━━━ 2:37:49 28s/step - accuracy: 0.7500 - loss: 0.7785

I0000 00:00:1772753130.020169     140 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


342/342 ━━━━━━━━━━━━━━━━━━━━ 147s 349ms/step - accuracy: 0.7077 - loss: 0.7951 - val_accuracy: 0.8734 - val_loss: 0.3496 - learning_rate: 3.0000e-04
Epoch 2/14
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 240ms/step - accuracy: 0.8857 - loss: 0.4035 - val_accuracy: 0.8734 - val_loss: 0.3141 - learning_rate: 3.0000e-04
Epoch 3/14
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 238ms/step - accuracy: 0.9099 - loss: 0.3323 - val_accuracy: 0.8580 - val_loss: 0.3219 - learning_rate: 3.0000e-04
Epoch 4/14
342/342 ━━━━━━━━━━━━━━━━━━━━ 81s 237ms/step - accuracy: 0.9134 - loss: 0.3153 - val_accuracy: 0.9056 - val_loss: 0.2584 - learning_rate: 3.0000e-04
Epoch 5/14
342/342 ━━━━━━━━━━━━━━━━━━━━ 81s 238ms/step - accuracy: 0.9225 - loss: 0.2888 - val_accuracy: 0.8748 - val_loss: 0.3354 - learning_rate: 3.0000e-04
Epoch 6/14
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 238ms/step - accuracy: 0.9281 - loss: 0.2771 - val_accuracy: 0.8675 - val_loss: 0.3426 - learning_rate: 3.0000e-04
Epoch 7/14
342/342 ━━━━━━━━━━━━━━━━━━━━ 81s 237ms/step -

...

342/342 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step - accuracy: 0.7215 - loss: 0.7047

2026-03-05 23:52:51.583335: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-05 23:52:51.724224: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-05 23:52:52.052572: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-05 23:52:52.192745: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-05 23:52:52.898292: E external/local_xla/xla/stream_

342/342 ━━━━━━━━━━━━━━━━━━━━ 120s 276ms/step - accuracy: 0.7217 - loss: 0.7043 - val_accuracy: 0.8741 - val_loss: 0.4035 - learning_rate: 3.0000e-04
Epoch 2/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 66s 192ms/step - accuracy: 0.8835 - loss: 0.4113 - val_accuracy: 0.8719 - val_loss: 0.3548 - learning_rate: 3.0000e-04
Epoch 3/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 66s 191ms/step - accuracy: 0.9039 - loss: 0.3475 - val_accuracy: 0.8712 - val_loss: 0.3439 - learning_rate: 3.0000e-04
Epoch 4/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 66s 192ms/step - accuracy: 0.9132 - loss: 0.3397 - val_accuracy: 0.8697 - val_loss: 0.3521 - learning_rate: 3.0000e-04
Epoch 5/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 66s 192ms/step - accuracy: 0.9208 - loss: 0.2986 - val_accuracy: 0.8770 - val_loss: 0.3353 - learning_rate: 3.0000e-04
Epoch 6/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 65s 190ms/step - accuracy: 0.9253 - loss: 0.3002 - val_accuracy: 0.8895 - val_loss: 0.3173 - learning_rate: 3.0000e-04
Epoch 7/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 65s 191ms/step -

2026-03-06 00:13:09.914580: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-06 00:13:10.052245: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-06 00:13:10.365768: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-06 00:13:10.505977: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-06 00:13:11.202439: E external/local_xla/xla/stream_


[EfficientNetB0] Test ACC=0.9148 | F1(macro)=0.8171
              precision    recall  f1-score   support

      Normal       0.88      0.82      0.85       898
 Hemorrhagic       0.55      0.71      0.62       228
    Ischemic       0.98      0.98      0.98      2053

    accuracy                           0.91      3179
   macro avg       0.80      0.84      0.82      3179
weighted avg       0.92      0.91      0.92      3179

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

[ResNet50V2] Stage-1
Epoch 1/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 107s 275ms/step - accuracy: 0.7477 - loss: 0.7256 - val_accuracy: 0.8250 - val_loss: 0.4622 - learning_rate: 3.0000e-04
Epoch 2/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 239ms/step - accuracy: 0.8962 - loss: 0.3690 - val_accuracy: 0.8258 - val_loss: 0.4803 - learning_rate: 3.0000e-04
Epoch 3/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 239ms/step - accuracy: 0.9100 - loss: 0.3300 - val_accuracy: 0.8360 - val_loss: 0.4197 - learning_rate: 3.0000e-04
Epoch 4/12
342/

In [18]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [19]:
SEED = 42
IMG_SIZE = (224,224)
BATCH_SIZE = 32
tf.random.set_seed(SEED)

In [20]:
def patient_wise_split(df, test_frac=0.20, val_frac=0.10, seed=42):

    pl = df.groupby("patient_id")["label"].agg(lambda x: x.mode()[0]).reset_index()

    pids = pl.patient_id.values
    plabs = pl.label.values

    p_train, p_temp, _, l_temp = train_test_split(
        pids,
        plabs,
        test_size=test_frac + val_frac,
        stratify=plabs,
        random_state=seed
    )

    vf = val_frac/(val_frac+test_frac)

    p_val, p_test = train_test_split(
        p_temp,
        test_size=1-vf,
        stratify=l_temp,
        random_state=seed
    )

    train_df = df[df.patient_id.isin(p_train)].reset_index(drop=True)
    val_df   = df[df.patient_id.isin(p_val)].reset_index(drop=True)
    test_df  = df[df.patient_id.isin(p_test)].reset_index(drop=True)

    print("Train:",len(train_df))
    print("Val:",len(val_df))
    print("Test:",len(test_df))

    print(train_df.label.value_counts())

    return train_df,val_df,test_df


train_df,val_df,test_df = patient_wise_split(df_all)

Train: 10939
Val: 1366
Test: 3179
label
2    7305
0    3037
1     597
Name: count, dtype: int64


In [21]:
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(train_df.label.unique())),
    y=train_df.label.values
)

class_weights = dict(zip(sorted(train_df.label.unique()),cw))

print(class_weights)

{np.int64(0): np.float64(1.2006365931291845), np.int64(1): np.float64(6.107761027359017), np.int64(2): np.float64(0.4991558293406343)}


In [22]:
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomContrast(0.08),
])

In [23]:
def parse_fn(fp,label,training=False):

    img = tf.io.read_file(fp)
    img = tf.image.decode_image(img,channels=3)

    img = tf.image.resize(img,IMG_SIZE)

    img = tf.cast(img,tf.float32)/255.0

    if training:
        img = aug(img)

    return img,tf.cast(label,tf.int32)

In [25]:
def parse_fn(fp, label, training=False):

    img = tf.io.read_file(fp)

    img = tf.image.decode_jpeg(img, channels=3)

    img = tf.image.resize(img, IMG_SIZE)

    img = tf.cast(img, tf.float32) / 255.0

    if training:
        img = aug(img)

    return img, tf.cast(label, tf.int32)

In [26]:
train_ds = make_ds(train_df, True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

In [27]:
for img, label in train_ds.take(1):
    print(img.shape)
    print(label.shape)

(32, 224, 224, 3)
(32,)


In [28]:
def cls_head(x,n_classes=3):

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(0.4)(x)

    out = layers.Dense(n_classes,activation="softmax")(x)

    return out

In [29]:
def build_densenet():

    base = tf.keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=(224,224,3)
    )

    base.trainable=False

    inp = Input(shape=(224,224,3))

    x = tf.keras.applications.densenet.preprocess_input(inp*255.0)

    x = base(x)

    out = cls_head(x)

    model = Model(inp,out)

    model.compile(
        optimizer=Adam(3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model,base

In [30]:
def build_efficient():

    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224,224,3)
    )

    base.trainable=False

    inp = Input(shape=(224,224,3))

    x = base(inp*255.0)

    out = cls_head(x)

    model = Model(inp,out)

    model.compile(
        optimizer=Adam(3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model,base

In [31]:
def build_resnet():

    base = tf.keras.applications.ResNet50V2(
        include_top=False,
        weights="imagenet",
        input_shape=(224,224,3)
    )

    base.trainable=False

    inp = Input(shape=(224,224,3))

    x = tf.keras.applications.resnet_v2.preprocess_input(inp*255.0)

    x = base(x)

    out = cls_head(x)

    model = Model(inp,out)

    model.compile(
        optimizer=Adam(3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model,base

In [32]:
def train_model(builder):

    model,base = builder()

    cb = [
        EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2
        )
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=12,
        class_weight=class_weights,
        callbacks=cb
    )

    return model

In [33]:
m1 = train_model(build_densenet)

m2 = train_model(build_efficient)

m3 = train_model(build_resnet)

Epoch 1/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 132s 321ms/step - accuracy: 0.6960 - loss: 0.7877 - val_accuracy: 0.8346 - val_loss: 0.3963 - learning_rate: 3.0000e-04
Epoch 2/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 239ms/step - accuracy: 0.8722 - loss: 0.4245 - val_accuracy: 0.8470 - val_loss: 0.3513 - learning_rate: 3.0000e-04
Epoch 3/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 239ms/step - accuracy: 0.9041 - loss: 0.3453 - val_accuracy: 0.8865 - val_loss: 0.2900 - learning_rate: 3.0000e-04
Epoch 4/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 239ms/step - accuracy: 0.9218 - loss: 0.2992 - val_accuracy: 0.8917 - val_loss: 0.2841 - learning_rate: 3.0000e-04
Epoch 5/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 239ms/step - accuracy: 0.9156 - loss: 0.2953 - val_accuracy: 0.9004 - val_loss: 0.2752 - learning_rate: 3.0000e-04
Epoch 6/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 81s 238ms/step - accuracy: 0.9271 - loss: 0.2653 - val_accuracy: 0.8594 - val_loss: 0.3255 - learning_rate: 3.0000e-04
Epoch 7/12
342/342 ━━━━━━━━━━━━━━━━━━━━ 82s 2

In [34]:
def ensemble_predict(models,ds):

    y_true=[]
    y_pred=[]

    for x,y in ds:

        preds=[m.predict(x,verbose=0) for m in models]

        avg=np.mean(preds,axis=0)

        p=np.argmax(avg,axis=1)

        y_true.extend(y.numpy())
        y_pred.extend(p)

    return np.array(y_true),np.array(y_pred)

In [35]:
models=[m1,m2,m3]

y_true,y_pred = ensemble_predict(models,test_ds)

print("Accuracy:",accuracy_score(y_true,y_pred))

print("F1:",f1_score(y_true,y_pred,average="macro"))

print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal","Hemorrhagic","Ischemic"]
))

Accuracy: 0.9402327776030198
F1: 0.8560157254472264
              precision    recall  f1-score   support

      Normal       0.94      0.85      0.89       898
 Hemorrhagic       0.60      0.79      0.68       228
    Ischemic       0.99      1.00      0.99      2053

    accuracy                           0.94      3179
   macro avg       0.84      0.88      0.86      3179
weighted avg       0.95      0.94      0.94      3179

